In [1]:
import os
os.environ["HF_HOME"] = os.path.expanduser("/home/s.dalal.800/.cache/huggingface")
os.environ["HF_DATASETS_CACHE"] = os.path.expanduser("/home/s.dalal.800/.cache/huggingface/datasets")

In [2]:
from datasets import load_dataset, load_from_disk, Audio
from transformers import Trainer, TrainingArguments
import torch

from safetensors.torch import load_file

In [3]:
from pyha_analyzer.models.efficientnet import EfficientNet
from pyha_analyzer.metrics.classification_metrics import AudioClassificationMetrics
from pyha_analyzer import PyhaTrainer, PyhaTrainingArguments
from pyha_analyzer.extractors import samExtractor, Birdset
from pyha_analyzer.preprocessors.birdset_spectrogram_preprocessors import BirdSetSpectrogramPreprocessor

In [4]:
data_dir = "/home/s.dalal.800/mount/SAM/HSN/test_5s/sam-audio-base/32k/test"

In [4]:
### RUN THIS CELL IF YOU WANT TO GET BASELINE RESULTS
birdset_extractor = Birdset()
test_data = birdset_extractor(region="HSN")

In [5]:
### RUN THIS CELL IF YOU WANT TO GET SAM RESULTS

SamExtractor = samExtractor.SamExtractor()

test_data = SamExtractor(data_dir)

Resolving data files:   0%|          | 0/12001 [00:00<?, ?it/s]

Computing checksums: 100%|#########9| 11990/12001 [00:05<00:00, 2397.88it/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
        num_rows: 5460
    })
    test: Dataset({
        features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
        num_rows: 10296
    })
    test_5s: Dataset({
        features: [

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

------------------
{'audio': {'bytes': None, 'path': '/home/s.dalal.800/mount/SAM/HSN/test_5s/sam-audio-base/32k/test/sam_HSN_001_20150708_061805_000_005.ogg'}, 'start_time': 0.0, 'end_time': 5.0, 'low_freq': None, 'high_freq': None, 'ebird_code': None, 'labels': [0], 'ebird_code_secondary': None, 'call_type': None, 'sex': None, 'lat': 37.0, 'long': -118.5, 'length': None, 'microphone': 'Soundscape', 'license': 'Creative Commons Attribution 4.0 International Public License', 'source': 'https://zenodo.org/record/7525805', 'local_time': '6:18:08', 'detected_events': None, 'event_cluster': None, 'peaks': None, 'quality': None, 'recordist': None, 'genus': None, 'species_group': None, 'order': None, 'genus_multilabel': '[10]', 'species_group_multilabel': '[1]', 'order_multilabel': '[2]', 'audio_in': {'bytes': None, 'path': '/home/s.dalal.800/mount/SAM/HSN/test_5s/sam-audio-base/32k/test/sam_HSN_001_20150708_061805_000_005.ogg'}, 'filepath': '/home/s.dalal.800/mount/SAM/HSN/test_5s/sam-audio

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/12000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'labels', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel', 'audio_in', 'filepath'],
        num_rows: 12000
    })
})
{'audio': {'bytes': None, 'path': '/home/s.dalal.800/mount/SAM/HSN/test_5s/sam-audio-base/32k/test/sam_HSN_001_20150708_061805_000_005.ogg'}, 'start_time': 0.0, 'end_time': 5.0, 'low_freq': None, 'high_freq': None, 'ebird_code': None, 'labels': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'ebird_code_secondary': None, 'call_type': None, 'sex': None, 'lat': 37.0, 'long': -118.5, 'length': None, 'microphone': 'Soundscape', 'license': 'Creative Commons Attribution 4.0 International Public Li

In [6]:
test_data

DatasetDict({
    train: Dataset({
        features: ['audio', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'labels', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel', 'audio_in', 'filepath'],
        num_rows: 12000
    })
})

In [7]:
preprocessor = BirdSetSpectrogramPreprocessor()
test_data["train"].set_transform(preprocessor)

In [8]:
num_classes = 21
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
model = EfficientNet(num_classes=num_classes).to(device)

In [10]:
model_name = "30-HSN-no-aug"    

In [11]:
model_state_dict = load_file(f"/home/s.dalal.800/models/HSN_BirdSet/with_aug/checkpoint-6750/model.safetensors")
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [12]:
sam = False

In [13]:
sam_name = "sam" if sam else "no-sam"
run_name = f"{model_name}-{sam_name}"

In [14]:
training_args = TrainingArguments()

In [15]:
training_args.per_device_eval_batch_size = 64
training_args.dataloader_num_workers = 16
training_args.remove_unused_columns = False

In [16]:
compute_metrics = AudioClassificationMetrics([], num_classes=num_classes)

In [17]:
model.eval()

trainer = Trainer(
    args=training_args,
    model=model,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
    data_collator=None,
)

In [18]:
training_args = PyhaTrainingArguments(
    working_dir="pyha-analyzer-2.0"
)

training_args.per_device_eval_batch_size = 64
training_args.dataloader_num_workers = 16
training_args.remove_unused_columns = True

In [19]:

def collate_fn(features):
    return {
        "audio_in": torch.stack([torch.as_tensor(f["audio_in"]) for f in features]),
        "labels":   torch.stack([torch.as_tensor(f["labels"])   for f in features]),
    }

In [20]:
### IF YOU ARE TRYING TO GET BASE RESULTS

model.eval()
trainer = PyhaTrainer(
    model=model,
    dataset=test_data,
    metrics=compute_metrics,
    training_args=training_args,
    # data_collator=collate_fn,
)

KeyError: 'valid'

In [22]:
test_data["train"]

Dataset({
    features: ['audio', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'labels', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel', 'audio_in', 'filepath'],
    num_rows: 12000
})

In [23]:
test_data["train"][0]

{'audio': array([[[ -5.023841 ,  -5.3149652,  -5.7816634, ...,  -1.960886 ,
           -2.2033756,  -4.151472 ],
         [ -3.853684 ,  -4.1448083,  -4.6115065, ...,  -0.790729 ,
           -1.0332185,  -2.9813154],
         [ -5.2496753,  -7.194511 ,  -6.2217956, ...,  -3.8721433,
           -3.0879304,  -3.1618984],
         ...,
         [ -8.618392 ,  -8.923891 ,  -9.0789175, ..., -11.147345 ,
          -11.326676 , -10.820873 ],
         [ -9.394947 , -10.42562  , -11.218987 , ..., -12.447163 ,
          -12.3126135, -11.1076765],
         [-12.422774 , -14.127258 , -15.217579 , ..., -15.217579 ,
          -13.202082 , -10.99115  ]]], shape=(1, 128, 501), dtype=float32),
 'start_time': 0.0,
 'end_time': 5.0,
 'low_freq': None,
 'high_freq': None,
 'ebird_code': None,
 'labels': array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], dtype=float32),
 'ebird_code_secondary': None,
 'call_type': None,
 'sex': None,
 'lat': 37.0,
 'long': -

In [24]:
results = trainer.evaluate(eval_dataset=test_data["train"], metric_key_prefix="Soundscape")

/home/s.dalal.800/pyha-analyzer-2.0/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.